# Upload dtsupply.us images to Cloudflare R2 (v2 — fuller headers)

**Fix:** a plain `Referer` header alone got 403 Forbidden — their CDN checks more browser-like signals. This version adds `Origin`, `Accept`, `Sec-Fetch-*` headers to look more like a real browser image request.

Reads `dtsupply_products.csv`, downloads each image, uploads it to your R2 bucket (`dtsupply-products/` folder), and writes `dtsupply_products_r2.csv`.

**Before running:** delete or rename any partial `dtsupply_products_r2.csv` from the failed attempt first — otherwise rows already "processed" (even though the image failed) won't be retried. Run the cell below to clear it.

In [1]:
import os
if os.path.exists("dtsupply_products_r2.csv"):
    os.remove("dtsupply_products_r2.csv")
    print("Old partial CSV deleted — ready for a fresh run.")
else:
    print("No existing file found — starting fresh.")

Old partial CSV deleted — ready for a fresh run.


In [2]:
!pip install requests boto3

In [ ]:
#!/usr/bin/env python3
"""
Takes the image URLs from dtsupply_products.csv, downloads each image
(using a Referer header so their CDN's hotlink protection doesn't block us),
uploads it to Cloudflare R2, and writes a NEW CSV with the "image" column
replaced by the new R2-hosted URL.

Requirements:
    pip install requests boto3

Output:
    dtsupply_products_r2.csv

Resumable: if you stop it and rerun, it skips products (by product_id)
already present in the output CSV.
"""

import csv
import os
import re
import time
import hashlib
from urllib.parse import urlparse

import requests
import boto3
from botocore.config import Config

# ---------------------------------------------------------------------------
# CLOUDFLARE R2 SETTINGS — reusing the same bucket/account from before
# ---------------------------------------------------------------------------
CLOUDFLARE_ACCOUNT_ID = "d42f7c5ed83f1403699b96fc13759c01"
R2_ACCESS_KEY_ID = "93510dc42d0c272e24a95c87f7a20313"
R2_SECRET_ACCESS_KEY = "38de75f0cdb543d0e6f81811faab924e31ec6116cc95edb5d6e5e094a7b11241"
R2_BUCKET_NAME = "syntexserver-2003"
R2_PUBLIC_URL_BASE = "https://pub-4ae70fe1d6cf4e5ea327e14b66c0ad8e.r2.dev"

# ---------------------------------------------------------------------------
INPUT_CSV = "dtsupply_products.csv"
OUTPUT_CSV = "dtsupply_products_r2.csv"
REQUEST_DELAY = 0.3
TIMEOUT = 20
PROGRESS_EVERY = 25
KEY_PREFIX = "dtsupply-products/"   # separate folder inside the bucket, keeps these apart from the vertexnetworking images

CONTENT_TYPES = {
    ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    ".png": "image/png", ".webp": "image/webp",
    ".gif": "image/gif", ".bmp": "image/bmp",
}

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    # Their CDN blocks hotlinking unless the request looks like it came
    # from the dtsupply.us site itself. A plain Referer alone wasn't
    # enough, so this mimics a real browser's image-loading request more
    # closely (Origin + Sec-Fetch-* + Accept are commonly checked too).
    "Referer": "https://dtsupply.us/",
    "Origin": "https://dtsupply.us",
    "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Sec-Fetch-Dest": "image",
    "Sec-Fetch-Mode": "no-cors",
    "Sec-Fetch-Site": "cross-site",
})

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    config=Config(signature_version="s3v4"),
    region_name="auto",
)


def safe_filename(url, sku=""):
    parsed = urlparse(url)
    ext = os.path.splitext(parsed.path)[1].lower()
    if ext not in CONTENT_TYPES:
        ext = ".jpg"
    base = (sku or "").strip()
    if not base:
        base = hashlib.md5(url.encode("utf-8")).hexdigest()[:12]
    base = re.sub(r"[^A-Za-z0-9_\-]", "_", base)
    return f"{base}{ext}"


def download_image(url):
    try:
        resp = session.get(url, timeout=TIMEOUT)
        resp.raise_for_status()
        return resp.content
    except requests.RequestException as e:
        print(f"  [warn] failed to download {url}: {e}")
        return None


def upload_to_r2(content, key, content_type):
    try:
        s3.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=key,
            Body=content,
            ContentType=content_type,
        )
        return f"{R2_PUBLIC_URL_BASE.rstrip('/')}/{key}"
    except Exception as e:
        print(f"  [warn] failed to upload {key} to R2: {e}")
        return None


def load_already_done():
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                pid = row.get("product_id")
                image = (row.get("image") or "").strip()
                if pid and (not image or image.startswith(R2_PUBLIC_URL_BASE)):
                    done.add(str(pid))
    return done


def main():
    if not os.path.exists(INPUT_CSV):
        raise SystemExit(f"Can't find {INPUT_CSV} — put this notebook in the same folder.")

    already_done = load_already_done()
    if already_done:
        print(f"Resuming — {len(already_done)} products already done in {OUTPUT_CSV}, will skip those.")

    with open(INPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    total = len(rows)
    print(f"Loaded {total} rows from {INPUT_CSV}.")

    file_exists = os.path.exists(OUTPUT_CSV)
    out_f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(out_f, fieldnames=fieldnames)
    if not file_exists:
        writer.writeheader()
        out_f.flush()

    processed = len(already_done)

    try:
        for row in rows:
            product_id = str(row.get("product_id", ""))
            if product_id and product_id in already_done:
                continue

            new_row = dict(row)
            image_url = (row.get("image") or "").strip()

            if image_url:
                content = download_image(image_url)
                if content:
                    ext = os.path.splitext(urlparse(image_url).path)[1].lower()
                    content_type = CONTENT_TYPES.get(ext, "image/jpeg")
                    key = KEY_PREFIX + safe_filename(image_url, row.get("sku", ""))
                    new_url = upload_to_r2(content, key, content_type)
                    if new_url:
                        new_row["image"] = new_url
                    time.sleep(REQUEST_DELAY)

            writer.writerow(new_row)
            out_f.flush()
            processed += 1

            if processed % PROGRESS_EVERY == 0:
                print(f"  >>> progress: {processed}/{total} rows processed")

    except KeyboardInterrupt:
        print("\n[stopped by user] Progress so far is safely saved.")
    finally:
        out_f.close()

    print(f"\nDone. {processed}/{total} rows written to {OUTPUT_CSV}")
    print("(rerun this cell anytime to resume/finish remaining rows)")


if __name__ == "__main__":
    main()


Loaded 231685 rows from dtsupply_products.csv.
  [warn] failed to download https://cdn.cmshardware.com/images/Images/Products/00XL284.webp: 403 Client Error: Forbidden for url: https://cdn.cmshardware.com/images/Images/Products/00XL284.webp
  [warn] failed to download https://cdn.cmshardware.com/images/Images/Products/017PW.webp: 403 Client Error: Forbidden for url: https://cdn.cmshardware.com/images/Images/Products/017PW.webp
  [warn] failed to download https://cdn.cmshardware.com/images/Images/Products/056397-000.webp: 403 Client Error: Forbidden for url: https://cdn.cmshardware.com/images/Images/Products/056397-000.webp
  [warn] failed to download https://cdn.cmshardware.com/images/Images/Products/0DKY0.webp: 403 Client Error: Forbidden for url: https://cdn.cmshardware.com/images/Images/Products/0DKY0.webp
  [warn] failed to download https://cdn.cmshardware.com/images/Images/Products/0G017A.webp: 403 Client Error: Forbidden for url: https://cdn.cmshardware.com/images/Images/Products